## Case 01. Association Rules

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
import pyreadstat
import pickle, argparse
import polars as pl
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules,fpmax, fpgrowth
from mlxtend.preprocessing import TransactionEncoder
#
warnings.filterwarnings("ignore")
#Import all relevant libraries
import matplotlib.pyplot as plt


## Reading dataset

In [2]:
file = './trombo_dat2.pkl'
fd   = open(file,'rb')
patD = pickle.load(fd)
ldata= pickle.load(fd)
#

In [3]:
a = TransactionEncoder()
a_data = a.fit(ldata).transform(ldata)
df = pd.DataFrame(a_data,columns=a.columns_)
df = df.replace(False,0)
df = df.replace(True,1)
#
frequent_itemsets = fpgrowth(df, min_support=0.001, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.625426,(ana_dura:Buscada negativo)
1,0.489027,(sin_tvp_:TVP)
2,0.471321,(sexo:Mujer)
3,0.256448,(edadC:Senior)
4,0.133864,(fr_tvp_a:Sí)
...,...,...
2872,0.002273,"(tv_l_svc:Sí, sexo:Mujer, ana_dura:Buscada neg..."
2873,0.002142,"(tv_l_svc:Sí, sexo:Mujer, edadC:Joven, ana_dur..."
2874,0.001880,"(sin_tvp_:TVP, tv_l_svc:Sí, ana_dura:Buscada n..."
2875,0.001224,"(othr_TUS:Sí, tv_l_svc:Sí, sin_tvp_:TVP)"


In [4]:
df.shape

(22874, 22)

In [5]:
rules = association_rules(frequent_itemsets, metric='confidence',min_threshold=0.4)

In [6]:
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules["consequent_len"] = rules["consequents"].apply(lambda x: len(x))

In [7]:
rules = rules.sort_values(by='leverage', ascending = False)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len
3026,(fr_estro:Sí),"(sexo:Mujer, edadC:Joven)",0.112967,0.204687,0.100070,0.885836,4.327768,0.076947,6.966407,0.866860,1,2
3023,"(sexo:Mujer, edadC:Joven)",(fr_estro:Sí),0.204687,0.112967,0.100070,0.488894,4.327768,0.076947,1.735516,0.966831,2,1
3024,"(sexo:Mujer, fr_estro:Sí)",(edadC:Joven),0.109950,0.389350,0.100070,0.910139,2.337584,0.057261,6.795504,0.642894,2,1
3020,(fr_estro:Sí),(edadC:Joven),0.112967,0.389350,0.100988,0.893963,2.296037,0.057004,5.758827,0.636354,1,1
3019,(fr_estro:Sí),(sexo:Mujer),0.112967,0.471321,0.109950,0.973297,2.065040,0.056707,19.798640,0.581430,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
31,"(edadC:Senior, ana_dura:Buscada negativo)",(sexo:Hombre),0.170893,0.528679,0.077206,0.451778,0.854541,-0.013142,0.859727,-0.170333,2,1
16,(edadC:Senior),(sin_tvp_:TVP),0.256448,0.489027,0.110475,0.430788,0.880908,-0.014935,0.897684,-0.153847,1,1
1845,"(edadC:Joven, ana_dura:Buscada negativo)",(sexo:Hombre),0.230961,0.528679,0.104311,0.451637,0.854275,-0.017794,0.859506,-0.181544,2,1
17,(edadC:Senior),(sexo:Hombre),0.256448,0.528679,0.117120,0.456700,0.863851,-0.018459,0.867515,-0.174894,1,1


In [8]:
# rules[ (rules['antecedent_len'] >= 2) &
#        (rules['confidence'] > 0.75) &
#        (rules['lift'] > 1.2) ]

In [9]:
rules.loc[0,'consequents']

frozenset({'sin_tvp_:TVP'})

In [10]:
match = {'ana_dura:Buscada negativo', 'ana_dura:Buscada positivo'}
rules['ana_dura'] = rules['consequents'].explode().isin(match).groupby(level=0).any()

In [11]:
rules[(rules['ana_dura']==True) & (rules['consequent_len']==1) & (rules['antecedent_len'] >=3)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura
1834,"(sexo:Hombre, edadC:Joven, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.101950,0.374574,0.044242,0.433962,1.158550,0.006055,1.104920,0.152388,3,1,True
45,"(sexo:Mujer, sin_tvp_:EP, edadC:Senior)",(ana_dura:Buscada negativo),0.054079,0.625426,0.038515,0.712207,1.138754,0.004693,1.301539,0.128814,3,1,True
1031,"(edadC:Joven, sin_tvp_:TVP/EP, sexo:Hombre)",(ana_dura:Buscada positivo),0.031083,0.374574,0.015083,0.485232,1.295425,0.003440,1.214967,0.235368,3,1,True
509,"(sexo:Mujer, sin_tvp_:EP, fr_inmov:Sí)",(ana_dura:Buscada negativo),0.030996,0.625426,0.022733,0.733427,1.172684,0.003348,1.405147,0.151966,3,1,True
1775,"(edadC:Medio, sexo:Hombre, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.111131,0.374574,0.044811,0.403226,1.076492,0.003184,1.048012,0.079941,3,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1268,"(evn_reci:Sí, edadC:Joven, sin_tvp_:TVP)",(ana_dura:Buscada negativo),0.018143,0.625426,0.008962,0.493976,0.789823,-0.002385,0.740229,-0.213233,3,1,True
1266,"(evn_reci:Sí, edadC:Joven, sexo:Hombre)",(ana_dura:Buscada negativo),0.020285,0.625426,0.010011,0.493534,0.789117,-0.002675,0.739584,-0.214314,3,1,True
1767,"(edadC:Medio, sin_tvp_:TVP, sexo:Hombre)",(ana_dura:Buscada negativo),0.111131,0.625426,0.066320,0.596774,0.954188,-0.003184,0.928943,-0.051246,3,1,True
1029,"(edadC:Joven, sin_tvp_:TVP/EP, sexo:Hombre)",(ana_dura:Buscada negativo),0.031083,0.625426,0.016001,0.514768,0.823067,-0.003440,0.771948,-0.181578,3,1,True


## Metrics for Evaluating Association Rules
In association rule mining, several metrics are commonly used to evaluate the quality and importance of the discovered association rules. 

These metrics can be used to evaluate the quality and importance of association rules and to select the most relevant rules for a given application. It is important to note that the appropriate choice of metric will depend on the specific goals and requirements of the application.

Interpreting the results of association rule mining metrics requires understanding the meaning and implications of each metric, as well as how to use them to evaluate the quality and importance of the discovered association rules. Here are some guidelines for interpreting the results of the main association rule mining metrics:

### Support
Support is a measure of how frequently an item or itemset appears in the dataset. It is calculated as the number of transactions containing the item(s) divided by the total number of transactions in the dataset. High support indicates that an item or itemset is common in the dataset, while low support indicates that it is rare.

Support Formula
![support.png](support.png)

### Confidence
Confidence is a measure of the strength of the association between two items. It is calculated as the number of transactions containing both items divided by the number of transactions containing the first item. High confidence indicates that the presence of the first item is a strong predictor of the presence of the second item.

Confidence Formula
![confidence.png](confidence.png)

### Lift
Lift is a measure of the strength of the association between two items, taking into account the frequency of both items in the dataset. It is calculated as the confidence of the association divided by the support of the second item. Lift is used to compare the strength of the association between two items to the expected strength of the association if the items were independent. 

A lift value greater than 1 indicates that the association between two items is stronger than expected based on the frequency of the individual items. This suggests that the association may be meaningful and worth further investigation. A lift value less than 1 indicates that the association is weaker than expected and may be less reliable or less significant.

Lift Formula

![lift.png](lift.png)


In [12]:
# Objetivo 1 => Perfil de Paciente con TROMBOFILIA POSITIVA => variable "ana_dura" 
# [No buscada, Buscada Positivo, Buscada Negativo.
# (factores de riesto, edad, sexo, tipo de cancer en su caso, etc.)
# Clasificación:
#    * Por localización: Usuales (Piernas, Pulmones) Inusuales (Brazos , cabeza, Ovarios) => >Tesis de LU
#    * Por causalidad: Provocada (cancer, embarazo, anticonceptivos)  No provocadas (No se conoce la causa) 
#  Hay mutacicón de genes y de factores de coaguliación (TODO ELLO LLAMADO TROMBOFILIA) que aumentan el riesgo de trombo
#  Las Trombofilias pueden ser genéticas o no y pueden ser adquiridas (debido a otras causas)
#   
#  El estudio de trombofilia cuando ya hay causalidad elevada no hace falta hacerlo porque no va a cambiar el tratamiento (anticoagulantes).
#  El estudio de trombofilia es (muy) caro 172 a 2400 USD)
#  No está claro cuando tiene sentido hacerlo.
#  Se decía que la trombosis en sitios raros hay que hacerlo siempre, pero si hay causas que ya implican riesgo => No hacerlo.
#  Los factores provocadores los hay mayores o menores El problema son los factores menores porque no está claro que justifique per sé el trombo.
#  Identificar a pacientes con riesgo 
#  Factores de Riesgo "variables" fr_*
#  fr_cance                                        No     (Si / NO al cancer)
#  fr_can_l                                       NaN     (Localizacion)
#  fr_can_e                                       NaN     (Estadío del cancer)
#   metacere                                       NaN    (Metástasis cerebrales) NaN => No
#  TVP => Trombosis Venosa Profunda
#

In [13]:
match = {'ana_dura:Buscada positivo'}
rules['ana_duraP'] = rules['consequents'].explode().isin(match).groupby(level=0).any()
rules[(rules['ana_duraP']==True)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP
1815,(edadC:Joven),(ana_dura:Buscada positivo),0.389350,0.374574,0.158389,0.406804,1.086046,0.012549,1.054334,0.129745,1,1,True,True
1828,"(edadC:Joven, sexo:Hombre)",(ana_dura:Buscada positivo),0.184664,0.374574,0.080353,0.435133,1.161674,0.011183,1.107209,0.170694,2,1,True,True
1819,"(sexo:Hombre, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.266329,0.374574,0.108158,0.406106,1.084183,0.008398,1.053095,0.105832,2,1,True,True
1824,"(edadC:Joven, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.209539,0.374574,0.086561,0.413102,1.102860,0.008073,1.065648,0.117990,2,1,True,True
1834,"(sexo:Hombre, edadC:Joven, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.101950,0.374574,0.044242,0.433962,1.158550,0.006055,1.104920,0.152388,3,1,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1914,"(edadC:Medio, evn_hemo:Sí, sin_tvp_:EP, sexo:M...",(ana_dura:Buscada positivo),0.003235,0.374574,0.001312,0.405405,1.082311,0.000100,1.051853,0.076298,4,1,True,True
2008,"(evn_reci:Sí, evn_hemo:Sí, edadC:Senior, sexo:...",(ana_dura:Buscada positivo),0.002885,0.374574,0.001180,0.409091,1.092150,0.000100,1.058414,0.084619,4,1,True,True
2188,"(evn_hemo:Sí, sin_tvp_:TVP/EP, edadC:Senior, s...",(ana_dura:Buscada positivo),0.002536,0.374574,0.001049,0.413793,1.104704,0.000099,1.066904,0.095021,4,1,True,True
171,"(edadC:Medio, fr_tvp_a:Sí, sin_tvp_:TVP, fr_in...",(ana_dura:Buscada positivo),0.002929,0.374574,0.001180,0.402985,1.075850,0.000083,1.047589,0.070709,4,1,True,True


In [14]:
sum(rules['ana_duraP']==True)

132

In [15]:
match = {'ana_dura:Buscada positivo', 'ana_dura:Buscada negativo'}
rules['ana_dura'] = rules['antecedents'].explode().isin(match).groupby(level=0).any()
# rules[(rules['ana_dura']==True) & (rules['consequent_len']==1) & (rules['antecedent_len'] >=3)]


In [16]:
rules[(rules['ana_dura']==True) & (rules['confidence'] > 0.85)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP
3039,"(ana_dura:Buscada negativo, fr_estro:Sí)","(sexo:Mujer, edadC:Joven)",0.069905,0.204687,0.061686,0.882427,4.311112,0.047377,6.764395,0.825766,2,2,True,False
3035,"(sexo:Mujer, ana_dura:Buscada negativo, fr_est...",(edadC:Joven),0.067981,0.389350,0.061686,0.907395,2.330537,0.035217,6.594168,0.612557,3,1,True,False
3031,"(ana_dura:Buscada negativo, fr_estro:Sí)",(edadC:Joven),0.069905,0.389350,0.062254,0.890557,2.287289,0.035037,5.579593,0.605100,2,1,True,False
3028,"(ana_dura:Buscada negativo, fr_estro:Sí)",(sexo:Mujer),0.069905,0.471321,0.067981,0.972483,2.063312,0.035034,19.212670,0.554075,2,1,True,False
3036,"(edadC:Joven, ana_dura:Buscada negativo, fr_es...",(sexo:Mujer),0.062254,0.471321,0.061686,0.990871,2.102326,0.032344,57.910668,0.559146,3,1,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4102,"(ana_dura:Buscada positivo, othr_TUS:Sí, fr_es...",(sexo:Mujer),0.001224,0.471321,0.001049,0.857143,1.818596,0.000472,3.700752,0.450677,3,1,True,False
3906,"(ana_dura:Buscada positivo, tv_l_esu:Sí, fr_in...",(sin_tvp_:TVP),0.001224,0.489027,0.001049,0.857143,1.752752,0.000451,3.576812,0.429995,3,1,True,False
3944,"(ana_dura:Buscada positivo, fr_cirug:Sí, tv_l_...",(sin_tvp_:TVP),0.001224,0.489027,0.001049,0.857143,1.752752,0.000451,3.576812,0.429995,3,1,True,False
3629,"(ana_dura:Buscada negativo, tv_l_esu:Sí, edadC...",(sin_tvp_:TVP),0.001137,0.489027,0.001006,0.884615,1.808930,0.000450,4.428434,0.447696,5,1,True,False


In [17]:
rules.columns

Index(['antecedents', 'consequents', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift', 'leverage',
       'conviction', 'zhangs_metric', 'antecedent_len', 'consequent_len',
       'ana_dura', 'ana_duraP'],
      dtype='object')

In [18]:
len(rules)

4805

In [19]:
# rules.loc[3926]['antecedents']

In [20]:
match = {'ana_dura:Buscada positivo','edadC:Senior'}
idx= rules['antecedents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP
34,"(ana_dura:Buscada positivo, edadC:Senior)",(sexo:Mujer),0.085556,0.471321,0.045641,0.533470,1.131860,0.005317,1.133214,0.127398,2,1,True,False
404,"(ana_dura:Buscada positivo, edadC:Senior, fr_i...",(sexo:Mujer),0.017181,0.471321,0.010449,0.608142,1.290293,0.002351,1.349161,0.228915,3,1,True,False
36,"(ana_dura:Buscada positivo, sin_tvp_:EP, edadC...",(sexo:Mujer),0.028067,0.471321,0.015564,0.554517,1.176517,0.002335,1.186755,0.154366,3,1,True,False
41,"(ana_dura:Buscada positivo, edadC:Senior, sin_...",(sexo:Mujer),0.039696,0.471321,0.020591,0.518722,1.100571,0.001882,1.098491,0.095158,3,1,True,False
409,"(ana_dura:Buscada positivo, edadC:Senior, sin_...",(sexo:Mujer),0.008612,0.471321,0.005508,0.639594,1.357024,0.001449,1.466898,0.265379,4,1,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40,"(ana_dura:Buscada positivo, sexo:Mujer, edadC:...",(sin_tvp_:TVP),0.045641,0.489027,0.020591,0.451149,0.922545,-0.001729,0.930988,-0.080859,3,1,True,False
38,"(ana_dura:Buscada positivo, edadC:Senior, sin_...",(sexo:Hombre),0.039696,0.528679,0.019105,0.481278,0.910340,-0.001882,0.908619,-0.093021,3,1,True,False
35,"(ana_dura:Buscada positivo, edadC:Senior)",(sin_tvp_:TVP),0.085556,0.489027,0.039696,0.463975,0.948773,-0.002143,0.953265,-0.055753,2,1,True,False
37,"(ana_dura:Buscada positivo, sin_tvp_:EP, edadC...",(sexo:Hombre),0.028067,0.528679,0.012503,0.445483,0.842634,-0.002335,0.849967,-0.161178,3,1,True,False


In [21]:
match = {'ana_dura:Buscada negativo','edadC:Senior'}
idx= rules['antecedents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP
19,"(edadC:Senior, ana_dura:Buscada negativo)",(sexo:Mujer),0.170893,0.471321,0.093687,0.548222,1.163160,0.013142,1.170218,0.169186,2,1,True,False
46,"(sexo:Mujer, edadC:Senior, ana_dura:Buscada ne...",(sin_tvp_:EP),0.093687,0.312582,0.038515,0.411106,1.315194,0.009230,1.167303,0.264430,3,1,True,False
47,"(sin_tvp_:EP, edadC:Senior, ana_dura:Buscada n...",(sexo:Mujer),0.064790,0.471321,0.038515,0.594467,1.261278,0.007979,1.303664,0.221505,3,1,True,False
415,"(edadC:Senior, ana_dura:Buscada negativo, fr_i...",(sin_tvp_:EP),0.039696,0.312582,0.017443,0.439427,1.405799,0.005035,1.226278,0.300593,3,1,True,False
402,"(edadC:Senior, ana_dura:Buscada negativo, fr_i...",(sexo:Mujer),0.039696,0.471321,0.023476,0.591410,1.254791,0.004767,1.293909,0.211448,3,1,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28,"(ana_dura:Buscada negativo, edadC:Senior, sexo...",(sin_tvp_:TVP),0.077206,0.489027,0.034144,0.442242,0.904331,-0.003612,0.916120,-0.102849,3,1,True,False
389,"(edadC:Senior, ana_dura:Buscada negativo, fr_i...",(sexo:Hombre),0.039696,0.528679,0.016219,0.408590,0.772852,-0.004767,0.796945,-0.234338,3,1,True,False
49,"(sin_tvp_:EP, edadC:Senior, ana_dura:Buscada n...",(sexo:Hombre),0.064790,0.528679,0.026274,0.405533,0.767069,-0.007979,0.792847,-0.245113,3,1,True,False
20,"(edadC:Senior, ana_dura:Buscada negativo)",(sin_tvp_:TVP),0.170893,0.489027,0.070779,0.414172,0.846932,-0.012792,0.872224,-0.178971,2,1,True,False


In [22]:
match = {'ana_dura:Buscada negativo','fr_antfa:Sí'}
idx= rules['antecedents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [23]:
idx2= rules['consequents'].apply(lambda consequent: all(item in consequent for item in match))
rules[idx2]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [24]:
match = {'ana_dura:Buscada negativo','fr_antfa:No'}
idx3= rules['antecedents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx3]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [25]:
idx4= rules['consequents'].apply(lambda consequent: all(item in consequent for item in match))
rules[idx4]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [26]:
match = {'ana_dura:Buscada positivo','fr_antfa:Sí'}
idx= rules['antecedents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [27]:
idx5= rules['consequents'].apply(lambda consequent: all(item in consequent for item in match))
rules[idx5]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [28]:
match = {'ana_dura:Buscada positivo','fr_antfa:No'}
idx6= rules['antecedents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx6]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [29]:
idx7= rules['consequents'].apply(lambda consequent: all(item in consequent for item in match))
rules[idx7]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP


In [30]:
match = {'ana_dura:Buscada positivo'}
idx8= rules['consequents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx8]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP
1815,(edadC:Joven),(ana_dura:Buscada positivo),0.389350,0.374574,0.158389,0.406804,1.086046,0.012549,1.054334,0.129745,1,1,False,True
1828,"(edadC:Joven, sexo:Hombre)",(ana_dura:Buscada positivo),0.184664,0.374574,0.080353,0.435133,1.161674,0.011183,1.107209,0.170694,2,1,False,True
1819,"(sexo:Hombre, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.266329,0.374574,0.108158,0.406106,1.084183,0.008398,1.053095,0.105832,2,1,False,True
1824,"(edadC:Joven, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.209539,0.374574,0.086561,0.413102,1.102860,0.008073,1.065648,0.117990,2,1,False,True
1834,"(sexo:Hombre, edadC:Joven, sin_tvp_:TVP)",(ana_dura:Buscada positivo),0.101950,0.374574,0.044242,0.433962,1.158550,0.006055,1.104920,0.152388,3,1,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1914,"(edadC:Medio, evn_hemo:Sí, sin_tvp_:EP, sexo:M...",(ana_dura:Buscada positivo),0.003235,0.374574,0.001312,0.405405,1.082311,0.000100,1.051853,0.076298,4,1,False,True
2008,"(evn_reci:Sí, evn_hemo:Sí, edadC:Senior, sexo:...",(ana_dura:Buscada positivo),0.002885,0.374574,0.001180,0.409091,1.092150,0.000100,1.058414,0.084619,4,1,False,True
2188,"(evn_hemo:Sí, sin_tvp_:TVP/EP, edadC:Senior, s...",(ana_dura:Buscada positivo),0.002536,0.374574,0.001049,0.413793,1.104704,0.000099,1.066904,0.095021,4,1,False,True
171,"(edadC:Medio, fr_tvp_a:Sí, sin_tvp_:TVP, fr_in...",(ana_dura:Buscada positivo),0.002929,0.374574,0.001180,0.402985,1.075850,0.000083,1.047589,0.070709,4,1,False,True


In [31]:
match = {'ana_dura:Buscada negativo'}
idx= rules['consequents'].apply(lambda antecedent: all(item in antecedent for item in match))
rules[idx]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedent_len,consequent_len,ana_dura,ana_duraP
3040,(fr_estro:Sí),"(sexo:Mujer, edadC:Joven, ana_dura:Buscada neg...",0.112967,0.126650,0.061686,0.546053,4.311497,0.047378,1.923901,0.865877,1,3,False,False
3037,"(sexo:Mujer, fr_estro:Sí)","(edadC:Joven, ana_dura:Buscada negativo)",0.109950,0.230961,0.061686,0.561034,2.429129,0.036292,1.751932,0.661008,2,2,False,False
3032,(fr_estro:Sí),"(edadC:Joven, ana_dura:Buscada negativo)",0.112967,0.230961,0.062254,0.551084,2.386047,0.036163,1.713101,0.654876,1,2,False,False
3029,(fr_estro:Sí),"(sexo:Mujer, ana_dura:Buscada negativo)",0.112967,0.304232,0.067981,0.601780,1.978031,0.033613,1.747196,0.557416,1,2,False,False
3038,"(edadC:Joven, fr_estro:Sí)","(sexo:Mujer, ana_dura:Buscada negativo)",0.100988,0.304232,0.061686,0.610823,2.007753,0.030962,1.787791,0.558314,2,2,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,"(sin_tvp_:TVP, sexo:Hombre)",(ana_dura:Buscada negativo),0.266329,0.625426,0.158171,0.593894,0.949582,-0.008398,0.922354,-0.067485,2,1,False,False
1,(sin_tvp_:TVP),(ana_dura:Buscada negativo),0.489027,0.625426,0.296581,0.606472,0.969694,-0.009269,0.951836,-0.057638,1,1,False,False
366,(sexo:Hombre),(ana_dura:Buscada negativo),0.528679,0.625426,0.321194,0.607542,0.971404,-0.009455,0.954429,-0.058786,1,1,False,False
1846,"(edadC:Joven, sexo:Hombre)",(ana_dura:Buscada negativo),0.184664,0.625426,0.104311,0.564867,0.903172,-0.011183,0.860827,-0.116210,2,1,False,False


## Más tests

In [32]:
rules.shape

(4805, 14)

In [33]:
rules.to_excel('reglas.xlsx')

In [ ]:
rules[ (rules['antecedent_len'] >= 2) & (rules['consequent_len'] >= 2) &
        (rules['confidence'] > 0.75) &
        (rules['lift'] > 1.2) ]

In [ ]:
rules[ (rules['antecedent_len'] >= 2) & (rules['consequent_len'] >= 2) &
        (rules['confidence'] > 0.75) &
        (rules['lift'] > 1.2) ].index.tolist()

In [ ]:
rules[ (rules['antecedent_len'] >= 2) & (rules['consequent_len'] >= 2) &
        (rules['confidence'] > 0.75) &
        (rules['lift'] > 1.2) ]

## LUCIA 21/01/2025

In [ ]:
match = {'ana_dura:Buscada positivo', 'ana_dura:Buscada negativo'}
match_raza = {'raza:Caucásica','raza:Asiática','raza:América Latina','raza:Arábica','raza:Otras'}
rules['ana_dura_A'] = rules['antecedents'].explode().isin(match).groupby(level=0).any()
# rules[(rules['ana_dura']==True) & (rules['consequent_len']==1) & (rules['antecedent_len'] >=3)]
rules['ana_dura_C'] = rules['consequents'].explode().isin(match).groupby(level=0).any()
# ahora la raza
# En la nueva versión raza ha sido barrida
rules['raza_A'] = rules['antecedents'].explode().isin(match_raza).groupby(level=0).any()
rules['raza_C'] = rules['consequents'].explode().isin(match_raza).groupby(level=0).any()

In [ ]:
idx_a1 = (rules['ana_dura_A']==True ) 
rules[idx_a1]

In [ ]:
idx_a2 = (rules['ana_dura_C']==True ) 
rules[idx_a2]

In [ ]:
rules[(rules['raza_A']==True)]

In [ ]:
idx_r1= rules['raza_A']==True
idx_r2= rules['raza_C']==True

In [ ]:
rules[(idx_a1) & (idx_r1)]

In [ ]:
rules.loc[641,'antecedents']

In [ ]:
rules[(idx_a1) & (idx_r2)]

In [ ]:
rules[(idx_a2) & (idx_r1)]

In [ ]:
rules[(idx_a2) & (idx_r2)]